# Etape 3 - Choix du dataset, des mod?les et benchmark

Ce notebook correspond ? l'?tape : **Choix du jeu de donn?es, du mod?le et de l'environnement exp?rimental**.

Objectif : comparer trois mod?les Hugging Face sur le m?me dataset afin de choisir le mod?le le plus adapt? pour la suite du stage.

La suite du stage ne consiste pas seulement ? trouver le mod?le avec la meilleure accuracy. Le vrai objectif est de choisir un mod?le r?aliste pour tester ensuite :

```text
FedAvg sans compression
FedAvg avec compression float16
FedAvg avec sparsification
FedAvg avec quantification ?ventuelle
```

Le benchmark doit donc comparer ? la fois :

- la qualit? du mod?le ;
- le temps d'entra?nement ;
- la taille du mod?le ;
- le volume de communication estim? ;
- la facilit? d'utilisation dans un cadre d'apprentissage f?d?r?.


## 1. Choix du dataset : Fashion-MNIST

Le dataset choisi est **Fashion-MNIST**.

Il contient des images de v?tements en noir et blanc de taille `28x28`. Il poss?de 10 classes : t-shirt, pantalon, pull, robe, manteau, sandale, chemise, sneaker, sac et bottine.

### Pourquoi ce dataset ?

Fashion-MNIST est adapt? ? ce stage parce qu'il est :

- simple ? charger avec `torchvision` ;
- assez petit pour lancer plusieurs exp?riences rapidement ;
- plus int?ressant que MNIST, car les classes sont moins faciles ? distinguer que des chiffres ;
- standard dans les projets de classification d'images ;
- suffisant pour comparer plusieurs mod?les et plusieurs strat?gies de compression.

### Limites du dataset

Fashion-MNIST reste un dataset simple. Les images sont petites, en niveaux de gris, et ne repr?sentent pas un vrai sc?nario industriel distribu?. Il est donc utilis? ici comme dataset exp?rimental pour comprendre le comportement des mod?les et des communications.


## 2. Mod?les choisis

Trois mod?les Hugging Face sont compar?s.

| Mod?le | Type | Pourquoi ce choix ? |
|---|---|---|
| `microsoft/resnet-18` | CNN classique | Mod?le l?ger, connu, bon point de d?part. |
| `facebook/convnext-tiny-224` | CNN moderne | Plus r?cent et plus performant, mais plus lourd. |
| `google/vit-base-patch16-224-in21k` | Vision Transformer | Architecture diff?rente, beaucoup plus lourde, int?ressante pour tester l'impact de la taille du mod?le. |

Ces mod?les sont plus complexes qu'un petit CNN ?crit ? la main. Ils permettent donc de mieux voir le lien entre taille du mod?le, performance et co?t de communication.


## 3. Avantages et inconv?nients des mod?les

### ResNet-18

**Avantages :**

- mod?le l?ger par rapport aux deux autres ;
- rapide ? entra?ner ;
- architecture CNN classique et facile ? expliquer ;
- bon choix pour commencer le benchmark Hugging Face ;
- co?t de communication plus faible car il a moins de param?tres.

**Inconv?nients :**

- architecture plus ancienne ;
- peut ?tre moins performant que des mod?les plus modernes ;
- il attend normalement des images RGB en `224x224`, donc il faut adapter Fashion-MNIST.

### ConvNeXT-Tiny

**Avantages :**

- CNN plus moderne ;
- souvent plus performant qu'un ResNet simple ;
- bon compromis entre performance et complexit? ;
- int?ressant pour tester un mod?le plus r?aliste.

**Inconv?nients :**

- plus lourd que ResNet-18 ;
- temps d'entra?nement plus long ;
- volume de communication plus important en apprentissage f?d?r?.

### ViT-Base

**Avantages :**

- architecture Vision Transformer, diff?rente des CNN ;
- tr?s utilis? dans les mod?les modernes de vision ;
- int?ressant pour observer l'effet d'un mod?le tr?s lourd sur la communication.

**Inconv?nients :**

- beaucoup plus lourd ;
- plus lent ? entra?ner ;
- co?t de communication ?lev? ;
- moins naturel pour Fashion-MNIST, car le dataset est petit et en noir et blanc ;
- probablement trop lourd pour les premi?res exp?riences FedAvg.


## 4. Environnement exp?rimental

Le benchmark utilise :

- Python ;
- PyTorch ;
- Torchvision pour Fashion-MNIST ;
- Transformers pour charger les mod?les Hugging Face ;
- GPU si disponible.

Par d?faut, le notebook utilise un **mode rapide** :

```python
FAST_MODE = True
```

Ce mode utilise un petit sous-ensemble du dataset pour v?rifier que le benchmark fonctionne. Quand tout marche, il faudra relancer avec :

```python
FAST_MODE = False
```

Pour acc?l?rer le benchmark, on entra?ne principalement la t?te de classification des mod?les :

```python
FREEZE_BACKBONE = True
```

Cela veut dire que le coeur du mod?le pr?-entra?n? est gel?, et que seule la partie finale est adapt?e aux 10 classes de Fashion-MNIST.


## 5. Installation des d?pendances

Lance cette cellule si les biblioth?ques n?cessaires ne sont pas encore install?es dans le kernel s?lectionn?.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = ["transformers", "tqdm"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    print("Installation des packages manquants:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("Toutes les d?pendances principales sont d?j? install?es.")


## 6. Imports et configuration

Tu peux commencer avec ces valeurs. Pour un premier test, garde `FAST_MODE = True`. Pour le vrai benchmark, mets `FAST_MODE = False`.


In [ ]:
import gc
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForImageClassification

SEED = 42
FAST_MODE = True

TRAIN_SUBSET_SIZE = 3000 if FAST_MODE else None
TEST_SUBSET_SIZE = 1000 if FAST_MODE else None

BATCH_SIZE = 16
NUM_EPOCHS = 1 if FAST_MODE else 3
LEARNING_RATE = 3e-5
FREEZE_BACKBONE = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("FAST_MODE:", FAST_MODE)
print("Epochs:", NUM_EPOCHS)


## 7. Chargement de Fashion-MNIST

Les images Fashion-MNIST sont en `28x28` et en niveaux de gris. Les mod?les Hugging Face attendent g?n?ralement des images RGB en `224x224`, donc les transformations seront adapt?es automatiquement pour chaque mod?le.


In [ ]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

label2id = {name: i for i, name in enumerate(class_names)}
id2label = {i: name for i, name in enumerate(class_names)}

base_transform = transforms.ToTensor()
train_base = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=base_transform)
test_base = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=base_transform)

generator = torch.Generator().manual_seed(SEED)
train_indices = torch.randperm(len(train_base), generator=generator)
test_indices = torch.randperm(len(test_base), generator=generator)

if TRAIN_SUBSET_SIZE is not None:
    train_indices = train_indices[:TRAIN_SUBSET_SIZE]
if TEST_SUBSET_SIZE is not None:
    test_indices = test_indices[:TEST_SUBSET_SIZE]

print("Train examples:", len(train_indices))
print("Test examples:", len(test_indices))


## 8. Visualisation du dataset

Cette cellule permet de v?rifier visuellement le dataset utilis?.


In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    image, label = train_base[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(image.squeeze(), cmap="gray")
    plt.title(class_names[label])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 9. Fonctions utiles

Ces fonctions servent ? pr?parer les images, charger les mod?les, compter les param?tres, estimer le volume de communication et entra?ner les mod?les.


In [ ]:
def get_processor_image_size(processor, default=224):
    crop_size = getattr(processor, "crop_size", None)
    if isinstance(crop_size, dict):
        if "height" in crop_size:
            return crop_size["height"]
        if "width" in crop_size:
            return crop_size["width"]

    size = getattr(processor, "size", default)
    if isinstance(size, dict):
        return size.get("height") or size.get("width") or size.get("shortest_edge") or default
    if isinstance(size, int):
        return size
    return default


def build_transform(processor):
    image_size = get_processor_image_size(processor)
    mean = getattr(processor, "image_mean", [0.485, 0.456, 0.406])
    std = getattr(processor, "image_std", [0.229, 0.224, 0.225])

    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


def build_dataloaders(processor):
    transform = build_transform(processor)

    train_dataset = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

    train_subset = Subset(train_dataset, train_indices.tolist())
    test_subset = Subset(test_dataset, test_indices.tolist())

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    return train_loader, test_loader


def freeze_backbone_keep_classifier(model):
    for param in model.parameters():
        param.requires_grad = False

    trainable_keywords = ["classifier", "head"]
    for name, param in model.named_parameters():
        if any(keyword in name.lower() for keyword in trainable_keywords):
            param.requires_grad = True


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def communication_sizes_mb(num_params):
    return {
        "float32_mb": num_params * 4 / (1024 ** 2),
        "float16_mb": num_params * 2 / (1024 ** 2),
        "int8_mb": num_params * 1 / (1024 ** 2),
    }


In [ ]:
def train_one_epoch(model, train_loader, optimizer):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for images, labels in tqdm(train_loader, desc="train", leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(pixel_values=images, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


def evaluate(model, test_loader):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    correct = 0

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="eval", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(pixel_values=images, labels=labels)
            loss = outputs.loss
            predictions = outputs.logits.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size
            correct += (predictions == labels).sum().item()

    return {
        "test_loss": total_loss / total_examples,
        "test_accuracy": correct / total_examples,
    }


## 10. Liste des mod?les ? tester

Chaque mod?le est accompagn? d'une justification courte. Le benchmark va ensuite mesurer ses performances r?elles sur Fashion-MNIST.


In [ ]:
models_to_test = [
    {
        "name": "ResNet-18",
        "model_id": "microsoft/resnet-18",
        "type": "CNN classique",
        "why": "Baseline Hugging Face l?g?re, connue et rapide.",
        "advantages": "L?ger, rapide, co?t de communication faible.",
        "limits": "Architecture plus ancienne, performance parfois inf?rieure aux mod?les modernes.",
    },
    {
        "name": "ConvNeXT-Tiny",
        "model_id": "facebook/convnext-tiny-224",
        "type": "CNN moderne",
        "why": "Mod?le plus moderne pour tester un compromis performance/co?t.",
        "advantages": "Plus performant qu'un CNN classique dans beaucoup de cas.",
        "limits": "Plus lourd, plus lent, communication plus co?teuse.",
    },
    {
        "name": "ViT-Base",
        "model_id": "google/vit-base-patch16-224-in21k",
        "type": "Vision Transformer",
        "why": "Architecture diff?rente et tr?s lourde pour mesurer l'effet de la taille du mod?le.",
        "advantages": "Tr?s moderne, puissant sur de grands datasets.",
        "limits": "Tr?s lourd, plus lent, moins adapt? ? un petit dataset comme Fashion-MNIST.",
    },
]

pd.DataFrame(models_to_test)


## 11. Benchmark des mod?les

Pour chaque mod?le, le notebook mesure :

- le nombre total de param?tres ;
- le nombre de param?tres entra?nables ;
- la loss d'entra?nement ;
- la loss de test ;
- l'accuracy de test ;
- le temps d'ex?cution ;
- le volume de communication estim? en `float32`, `float16` et `int8`.

Le volume de communication estim? correspond ? l'envoi d'une mise ? jour compl?te du mod?le par un client.


In [ ]:
def run_benchmark(model_info):
    model_id = model_info["model_id"]
    print("\n" + "=" * 80)
    print("Mod?le:", model_info["name"])
    print("Hugging Face ID:", model_id)
    print("Pourquoi ce mod?le:", model_info["why"])

    processor = AutoImageProcessor.from_pretrained(model_id)
    train_loader, test_loader = build_dataloaders(processor)

    model = AutoModelForImageClassification.from_pretrained(
        model_id,
        num_labels=10,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    if FREEZE_BACKBONE:
        freeze_backbone_keep_classifier(model)

    model = model.to(DEVICE)
    total_params, trainable_params = count_parameters(model)
    print(f"Param?tres totaux: {total_params:,}")
    print(f"Param?tres entra?nables: {trainable_params:,}")

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
    )

    start_time = time.perf_counter()
    train_losses = []

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        train_losses.append(train_loss)
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS} - train_loss={train_loss:.4f}")

    metrics = evaluate(model, test_loader)
    elapsed = time.perf_counter() - start_time
    sizes = communication_sizes_mb(total_params)

    result = {
        "name": model_info["name"],
        "model_id": model_id,
        "type": model_info["type"],
        "why": model_info["why"],
        "advantages": model_info["advantages"],
        "limits": model_info["limits"],
        "num_epochs": NUM_EPOCHS,
        "train_examples": len(train_loader.dataset),
        "test_examples": len(test_loader.dataset),
        "total_params": total_params,
        "trainable_params": trainable_params,
        "last_train_loss": train_losses[-1],
        "test_loss": metrics["test_loss"],
        "test_accuracy": metrics["test_accuracy"],
        "elapsed_seconds": elapsed,
        **sizes,
    }

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


In [ ]:
results = []

for model_info in models_to_test:
    try:
        result = run_benchmark(model_info)
        results.append(result)
    except Exception as exc:
        print(f"Erreur avec {model_info['name']}: {type(exc).__name__}: {exc}")
        results.append({
            "name": model_info["name"],
            "model_id": model_info["model_id"],
            "type": model_info["type"],
            "why": model_info["why"],
            "advantages": model_info["advantages"],
            "limits": model_info["limits"],
            "error": str(exc),
        })
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df


## 12. Sauvegarde des r?sultats

Les r?sultats sont sauvegard?s dans un fichier CSV pour pouvoir les r?utiliser dans le rapport.


In [ ]:
Path("results").mkdir(exist_ok=True)
output_path = Path("results/hf_model_benchmark_fashion_mnist.csv")
results_df.to_csv(output_path, index=False)
print("R?sultats sauvegard?s dans:", output_path)


## 13. Graphiques de comparaison

Les graphiques permettent de visualiser rapidement la diff?rence entre les mod?les.


In [ ]:
plot_df = results_df.dropna(subset=["test_accuracy"]).copy()

plt.figure(figsize=(8, 4))
plt.bar(plot_df["name"], plot_df["test_accuracy"])
plt.ylabel("Accuracy")
plt.title("Accuracy sur Fashion-MNIST")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
plot_df = results_df.dropna(subset=["float32_mb"]).copy()

x = range(len(plot_df))
width = 0.25

plt.figure(figsize=(9, 4))
plt.bar([i - width for i in x], plot_df["float32_mb"], width=width, label="float32")
plt.bar(x, plot_df["float16_mb"], width=width, label="float16")
plt.bar([i + width for i in x], plot_df["int8_mb"], width=width, label="int8")
plt.ylabel("Taille estim?e d'une mise ? jour compl?te (MB)")
plt.title("Volume de communication estim? par mod?le")
plt.xticks(list(x), plot_df["name"], rotation=20)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plot_df = results_df.dropna(subset=["elapsed_seconds"]).copy()

plt.figure(figsize=(8, 4))
plt.bar(plot_df["name"], plot_df["elapsed_seconds"])
plt.ylabel("Temps total (secondes)")
plt.title("Temps de benchmark par mod?le")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 14. Classement automatique provisoire

Ce classement est indicatif. Il combine trois crit?res :

- accuracy ?lev?e ;
- volume de communication faible ;
- temps d'ex?cution faible.

Les poids peuvent ?tre chang?s selon la priorit? du stage. Ici, on donne plus d'importance ? l'accuracy, mais on garde une p?nalisation pour les mod?les trop lourds.


In [ ]:
def normalize_higher_is_better(series):
    min_value = series.min()
    max_value = series.max()
    if max_value == min_value:
        return pd.Series([1.0] * len(series), index=series.index)
    return (series - min_value) / (max_value - min_value)


def normalize_lower_is_better(series):
    min_value = series.min()
    max_value = series.max()
    if max_value == min_value:
        return pd.Series([1.0] * len(series), index=series.index)
    return 1 - ((series - min_value) / (max_value - min_value))

ranking_df = results_df.dropna(subset=["test_accuracy", "float32_mb", "elapsed_seconds"]).copy()

ranking_df["accuracy_score"] = normalize_higher_is_better(ranking_df["test_accuracy"])
ranking_df["communication_score"] = normalize_lower_is_better(ranking_df["float32_mb"])
ranking_df["speed_score"] = normalize_lower_is_better(ranking_df["elapsed_seconds"])

ranking_df["final_score"] = (
    0.50 * ranking_df["accuracy_score"]
    + 0.30 * ranking_df["communication_score"]
    + 0.20 * ranking_df["speed_score"]
)

ranking_df = ranking_df.sort_values("final_score", ascending=False).reset_index(drop=True)
ranking_df["rank"] = ranking_df.index + 1

ranking_columns = [
    "rank", "name", "test_accuracy", "test_loss", "elapsed_seconds",
    "total_params", "float32_mb", "float16_mb", "int8_mb", "final_score"
]

ranking_df[ranking_columns]


## 15. Interpr?tation ? faire apr?s ex?cution

Apr?s avoir lanc? le notebook, il faudra regarder les r?sultats et choisir le mod?le le plus adapt? pour la suite.

Questions ? se poser :

- Quel mod?le a la meilleure accuracy ?
- Quel mod?le est le plus rapide ?
- Quel mod?le est le moins co?teux en communication ?
- Est-ce que le gain d'accuracy d'un mod?le lourd justifie son co?t ?
- Quel mod?le est r?aliste pour faire ensuite du FedAvg avec compression ?

Pour le stage, le meilleur choix n'est pas forc?ment le mod?le avec la meilleure accuracy. Un mod?le plus l?ger peut ?tre pr?f?rable s'il donne une accuracy correcte avec un volume de communication beaucoup plus faible.

Quand tu auras les r?sultats, envoie-moi le tableau `ranking_df` ou le fichier CSV, et je t'aiderai ? faire un classement argument? des mod?les.
